# Source balance v1 — CPU visual screening

This notebook checks eight **new training candidates** for the four source-confounded classes: dumbbell row, sit-up, lat pulldown, pull-up. Use **CPU** and **Run all**. It fetches a pinned, checksum-verified manifest, attempts 360p downloads, creates contact sheets, and returns one evidence ZIP. Failed YouTube access is recorded and does not abort the rest. No model training or labeling happens here; every source remains `training_allowed=false`. Raw video stays temporary in Colab and is excluded from the ZIP.


In [ ]:
%pip install -q --upgrade "yt-dlp[default]" opencv-python-headless pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib, json, platform, re, shutil, subprocess, urllib.request, zipfile
import cv2, numpy as np, pandas as pd
from google.colab import files
from IPython.display import display

PINNED_COMMIT = 'bb7fe7e63c299ba015246abb0405f011af6ab993'
RAW = f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}/'
INPUTS = {
 'ml/data/source_balance_v1/candidates.csv':'06cc829a21f7d91a02bea5913cabad06796050ec316a3037e5a829bc18e88cea',
 'ml/data/youtube_creator_benchmark_v1/candidates.csv':'aa3063810fa3b0e9fe169c04281ef65cc33a442d18be0fa0e5fec7669e06268b',
 'ml/data/youtube_creator_benchmark_v1/excluded_source_inventory.csv':'ecf18c29ef5b7ba1eca3f45677e463df256b965888f3913ae55606807baf2232',
 'ml/data/youtube_mixed_workouts_v1/manifest.csv':'1d58ed25261a172e49abe3f9b7313236d339e5fbe067ecccf838c6f187efb3eb',
}
WORK = Path('/content/calorie_source_balance_v1_work')
OUTPUT = Path('/content/calorie_source_balance_v1_screening')
for folder in (WORK,WORK/'videos',WORK/'frames',OUTPUT,OUTPUT/'contact_sheets',OUTPUT/'metadata'):
 folder.mkdir(parents=True,exist_ok=True)

def sha256(path):
 digest=hashlib.sha256()
 with Path(path).open('rb') as handle:
  for block in iter(lambda:handle.read(1024*1024),b''):digest.update(block)
 return digest.hexdigest()

def fetch_pinned(relative,digest):
 path=WORK/relative
 path.parent.mkdir(parents=True,exist_ok=True)
 if not path.exists() or sha256(path)!=digest:
  tmp=path.with_suffix(path.suffix+'.part')
  tmp.unlink(missing_ok=True)
  urllib.request.urlretrieve(RAW+relative,tmp)
  assert sha256(tmp)==digest,relative
  tmp.replace(path)
 return path

paths={name:fetch_pinned(name,digest) for name,digest in INPUTS.items()}
assert len({str(path) for path in paths.values()})==len(INPUTS), 'Pinned inputs share a cache path'
assert all(sha256(paths[name])==digest for name,digest in INPUTS.items())
manifest=pd.read_csv(paths['ml/data/source_balance_v1/candidates.csv'])
reserved=pd.concat([pd.read_csv(paths[name]) for name in INPUTS if name!='ml/data/source_balance_v1/candidates.csv'],ignore_index=True)
normalize=lambda value:re.sub('[^a-z0-9]+','',str(value).casefold())
assert manifest.video_id.is_unique and manifest.creator.map(normalize).is_unique
assert set(manifest.video_id).isdisjoint(set(reserved.video_id))
assert set(manifest.creator.map(normalize)).isdisjoint(set(reserved.creator.map(normalize)))
assert (manifest.status=='metadata_only').all() and (~manifest.training_allowed).all()
print('CPU screening:',len(manifest),'new creators; all training-locked')
display(manifest[['creator','target_classes','video_id']])


In [ ]:
def deno_args():
 arch=platform.machine().lower()
 asset={'x86_64':'deno-x86_64-unknown-linux-gnu.zip','aarch64':'deno-aarch64-unknown-linux-gnu.zip'}.get(arch)
 if asset is None:return []
 deno=WORK/'deno'/'deno'
 try:
  if not deno.exists():
   deno.parent.mkdir(parents=True,exist_ok=True)
   archive=deno.parent/'deno.zip'
   urllib.request.urlretrieve('https://github.com/denoland/deno/releases/latest/download/'+asset,archive)
   with zipfile.ZipFile(archive) as z:z.extract('deno',deno.parent)
   deno.chmod(0o755)
  subprocess.run([str(deno),'--version'],check=True,stdout=subprocess.DEVNULL,timeout=10)
  return ['--js-runtimes',f'deno:{deno}','--remote-components','ejs:github']
 except Exception as exc:
  print('Deno unavailable; yt-dlp will use its own fallback:',type(exc).__name__)
  return []

JS_ARGS=deno_args()

def command_result(args,timeout=240):
 try:
  result=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout,check=False)
  return result.returncode,result.stdout
 except subprocess.TimeoutExpired as exc:
  return 124,str(exc)

def media_info(path):
 code,out=command_result(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=width,height,duration','-of','json',str(path)],30)
 if code:return None
 try:
  stream=json.loads(out)['streams'][0]
  return {'width':int(stream['width']),'height':int(stream['height']),'duration_s':float(stream.get('duration') or 0)}
 except (KeyError,ValueError,IndexError):return None

def obtain_video(row):
 matches=[p for p in (WORK/'videos').glob(row.video_id+'.*') if p.suffix in {'.mp4','.mkv','.webm'} and media_info(p)]
 if matches:return matches[0],''
 failures=[]
 attempts=[('360p mp4','18/best[height=360][ext=mp4]'),('360p video','bestvideo[height=360]+bestaudio/best[height=360]')]
 for name,fmt in attempts:
  args=['yt-dlp','--no-playlist','--no-progress','--retries','2','--fragment-retries','2','--socket-timeout','20','-f',fmt,'--merge-output-format','mp4','-o',str(WORK/'videos'/(row.video_id+'.%(ext)s')),*JS_ARGS,row.source_url]
  code,out=command_result(args,300)
  matches=[p for p in (WORK/'videos').glob(row.video_id+'.*') if p.suffix in {'.mp4','.mkv','.webm'} and media_info(p)]
  if code==0 and matches:return matches[0],''
  failures.append(name+': '+' '.join(out.splitlines()[-5:])[-600:])
  if 'Sign in to confirm' in out or 'not a bot' in out:break
 return None,' | '.join(failures)

def contact_sheets(video_id,path,info):
 interval=5 if info['duration_s']<=900 else 10
 frame_dir=WORK/'frames'/video_id
 frame_dir.mkdir(parents=True,exist_ok=True)
 frame_pattern=str(frame_dir/'%06d.jpg')
 code,out=command_result(['ffmpeg','-hide_banner','-loglevel','error','-y','-i',str(path),'-vf',f'fps=1/{interval},scale=320:180:force_original_aspect_ratio=decrease,pad=320:180:(ow-iw)/2:(oh-ih)/2:black','-q:v','3',frame_pattern],1800)
 frames=sorted(frame_dir.glob('*.jpg'))
 if code or not frames:return 0,'ffmpeg failed: '+out[-500:]
 pages=0
 dest=OUTPUT/'contact_sheets'/video_id
 dest.mkdir(parents=True,exist_ok=True)
 for start in range(0,len(frames),12):
  tiles=[]
  for index in range(start,min(start+12,len(frames))):
   frame=cv2.imread(str(frames[index]))
   if frame is None:continue
   time_s=index*interval
   cv2.rectangle(frame,(0,0),(87,26),(0,0,0),-1)
   cv2.putText(frame,f'{time_s//60:02d}:{time_s%60:02d}',(5,18),cv2.FONT_HERSHEY_SIMPLEX,.55,(0,255,255),1)
   tiles.append(frame)
  if not tiles:continue
  while len(tiles)<12:tiles.append(np.zeros_like(tiles[0]))
  page=np.vstack([np.hstack(tiles[i:i+4]) for i in (0,4,8)])
  cv2.imwrite(str(dest/f'{pages:03d}.jpg'),page,[cv2.IMWRITE_JPEG_QUALITY,85])
  pages+=1
 return pages,''

results=[]
for row in manifest.itertuples(index=False):
 print('Screening',row.creator,row.video_id)
 record={'video_id':row.video_id,'creator':row.creator,'target_classes':row.target_classes,'status':'unavailable','pages':0,'height':None,'duration_s':None,'error':''}
 code,out=command_result(['yt-dlp','--skip-download','--dump-single-json','--no-warnings',*JS_ARGS,row.source_url],90)
 if code==0:
  try:
   raw=json.loads(out)
   metadata={key:raw.get(key) for key in ('id','title','channel','channel_id','duration','license','availability','webpage_url')}
   (OUTPUT/'metadata'/(row.video_id+'.json')).write_text(json.dumps(metadata,indent=2,ensure_ascii=False)+'\n')
   if metadata['id']!=row.video_id or normalize(metadata['channel']) in set(reserved.creator.map(normalize)):
    record['status']='identity_conflict';record['error']='Video or channel identity conflicts with reserved sources';results.append(record);continue
  except (ValueError,TypeError) as exc:
   record['error']='Metadata parse failed: '+str(exc)
 else:record['error']='Metadata unavailable: '+' '.join(out.splitlines()[-3:])[-400:]
 video,download_error=obtain_video(row)
 if video is None:
  record['error']=(record['error']+' | '+download_error).strip(' |');results.append(record);continue
 info=media_info(video)
 record.update(height=info['height'],duration_s=info['duration_s'])
 if info['height']!=360:
  record['status']='wrong_resolution';record['error']=f'Expected 360p, got {info["height"]}p';results.append(record);continue
 pages,error=contact_sheets(row.video_id,video,info)
 record.update(status='visual_evidence_ready' if pages else 'decode_failed',pages=pages,error=error)
 results.append(record)
 print(record['status'],'pages',pages)

status=pd.DataFrame(results)
status.to_csv(OUTPUT/'source_status.csv',index=False)
display(status[['creator','target_classes','status','pages','error']])


The contact sheets are **screening evidence only**. Review actual motion and boundaries independently before extracting landmarks or allowing training. A title, chapter, or metadata row is never a label.


In [ ]:
shutil.copy2(paths['ml/data/source_balance_v1/candidates.csv'],OUTPUT/'candidates.csv')
metadata={'notebook':'source_balance_v1_cpu_screening.ipynb','runtime':'CPU','pinned_commit':PINNED_COMMIT,'sources':len(manifest),'visual_evidence_ready':int((status.status=='visual_evidence_ready').sum()),'training_allowed':False,'raw_video_exported':False}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
lines=[]
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file()):lines.append(f'{sha256(path)}  {path.relative_to(OUTPUT)}')
(OUTPUT/'SHA256SUMS.txt').write_text('\n'.join(lines)+'\n')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
print('Created:',archive,round(Path(archive).stat().st_size/1e6,1),'MB')
files.download(archive)
